Из лекционной презентации: 
    
- **Алгоритм Нидлмана-Вунша** - шаги действий:

1) Инициализация. Создание матрицы выравнивания с размерами (n+1) x (m+1), где n и m — длины двух выравниваемых последовательностей. Инициализация первой строки и столбца матрицы штрафами за пробелы. 
2) Схема оценки. Определение схемы оценки, которая назначает баллы за совпадения, несовпадения и пробелы. Обычно за совпадения — положительные баллы, за несовпадения — отрицательные, а за пробелы — штрафы. 
3) Заполнение матрицы. Итерация по каждой клетке матрицы выравнивания, начиная с левого верхнего угла. Расчёт балла каждой клетки на основе трёх возможных движений: диагонального (совпадение/несовпадение), горизонтального (пробел в первой последовательности) и вертикального (пробел во второй последовательности). 
4) Трассировка. После заполнения матрицы трассировка по ней для определения оптимального пути выравнивания. Начинать с правого нижнего угла матрицы и следовать по пути с максимальным баллом обратно к левому верхнему углу. На каждом шаге решать направление движения (по диагонали, горизонтально или вертикально) на основе баллов соседних клеток. Записывать выровненные символы или пробелы по мере трассировки по матрице. 
5) Окончательное выравнивание. Путь, traced во время трассировки, представляет оптимальное выравнивание между двумя последовательностями. Извлечь выровненные символы или пробелы из трассировочного пути, чтобы получить окончательные выровненные последовательности.





In [1]:
def needleman_wunsch(seq1, seq2, match_score, mismatch_penalty, gap_penalty):
    """
    seq1, seq2 - выравниваемые последовательности нуклеотидов
    match_score - балл за совпадение
    mismatch_penalty - штраф за несовпадение
    gap_penalty - штраф за вставку или удаление
    """
    
    # инициализация матрицы из длин последовательностей.
    n = len(seq1)
    m = len(seq2)
    
    # создаём матрицу размера ((n+1) x (m+1)) для хранения баллов
    score_matrix = [[0 for j in range(m + 1)] for i in range(n + 1)]
    
    '''Инициализируем матрицу оценок размером ((n+1) x (m+1)) из нулей. 
    Эта матрица будет использоваться для хранения максимальных баллов выравнивания для каждой позиции.'''
    
    # создаём матрицу для хранения указателей на предыдущие клетки - для трассировки
    traceback_matrix = [['' for j in range(m + 1)] for i in range(n + 1)]
    
    '''Инициализируем матрицу трассировки того же размера для хранения указателей на предыдущие клетки. 
    Это поможет нам восстановить оптимальное выравнивание после заполнения матрицы оценок.'''
    
    # 1-я строка и 1-ый столбец матрицы заполняем штрафами за пробелы
    for i in range(1, n + 1):
        score_matrix[i][0] = gap_penalty * i
        traceback_matrix[i][0] = '↑'  # движение наверх
    for j in range(1, m + 1):
        score_matrix[0][j] = gap_penalty * j
        traceback_matrix[0][j] = '←'  # Обозначаем движение влево
    
    '''Каждый шаг вниз соответствует вставке пробела в seq2.
    Каждый шаг вправо соответствует вставке пробела в seq1.'''
    
    # заполнение матрицы оценок
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # совпадают ли текущие символы двух последовательностей
            if seq1[i - 1] == seq2[j - 1]:
                diag = score_matrix[i - 1][j - 1] + match_score
                
                '''Для каждой клетки [i][j] вычисляем балл диагонального движения (diag). 
            Если символы в последовательностях совпадают, добавляем match_score, иначе mismatch_penalty - штраф за несовпадение.'''
            else:
                diag = score_matrix[i - 1][j - 1] + mismatch_penalty
            up = score_matrix[i - 1][j] + gap_penalty   # вставка пробела в seq2
            left = score_matrix[i][j - 1] + gap_penalty  # вставка пробела в seq1
            
            
            '''баллы для вертикального (up) и горизонтального (left) движений + штраф за пробел.'''
            # выбираем максимальный балл из трех возможных движений
            max_score = max(diag, up, left)
            score_matrix[i][j] = max_score
            
            # заполняем матрицу трассировки
            if max_score == diag:
                traceback_matrix[i][j] = '↖'  # диагональное движение
            elif max_score == up:
                traceback_matrix[i][j] = '↑'  
            else:
                traceback_matrix[i][j] = '←' 
    
    '''Определяем, откуда пришел максимальный балл, и записываем соответствующий указатель в матрицу трассировки.'''
    
    # трассировка для получения оптимального выравнивания.
    align1 = ''
    align2 = ''
    i = n
    j = m
    
    '''переменные для хранения выровненных последовательностей и начинаем трассировку с нижнего правого угла матрицы.'''
    
    while i > 0 or j > 0:
        if traceback_matrix[i][j] == '↖':
            align1 = seq1[i - 1] + align1
            align2 = seq2[j - 1] + align2
            i -= 1
            j -= 1
        elif traceback_matrix[i][j] == '↑':
            align1 = seq1[i - 1] + align1
            align2 = '-' + align2
            i -= 1
        else:  
            align1 = '-' + align1
            align2 = seq2[j - 1] + align2
            j -= 1
    '''восстановливаем путь, по которому мы пришли к клетке [i][j]. В зависимости от указателя в матрице для трассировки, 
    определяем движение:'''
    
    # печатаем матрицу выравнивания
    print("Матрица выравнивания 2 последовательностей:")
    for row in score_matrix:
        print(row)
    
    # печатаем итоговое выравнивание
    print("\nИтоговое выравнивание - 2 последовательности с гэпами:")
    print(align1)
    print(align2)
    
    # печатаем скор выравнивания
    print("\nСкор выравнивания 2 последовательностей:", score_matrix[n][m])
    
    return align1, align2, score_matrix[n][m]

In [2]:
# Пример использования 1
seq1 = "GACGAAG"
seq2 = "ACCAAG"
match_score = 1
mismatch_penalty = -1
gap_penalty = -1

needleman_wunsch(seq1, seq2, match_score, mismatch_penalty, gap_penalty)

Матрица выравнивания 2 последовательностей:
[0, -1, -2, -3, -4, -5, -6]
[-1, -1, -2, -3, -4, -5, -4]
[-2, 0, -1, -2, -2, -3, -4]
[-3, -1, 1, 0, -1, -2, -3]
[-4, -2, 0, 0, -1, -2, -1]
[-5, -3, -1, -1, 1, 0, -1]
[-6, -4, -2, -2, 0, 2, 1]
[-7, -5, -3, -3, -1, 1, 3]

Итоговое выравнивание - 2 последовательности с гэпами:
GACGAAG
-ACCAAG

Скор выравнивания 2 последовательностей: 3


('GACGAAG', '-ACCAAG', 3)

In [3]:
# Пример использования 2
seq1 = "CGTCTT"
seq2 = "CATTCT"
match_score = 1
mismatch_penalty = -1
gap_penalty = -2

needleman_wunsch(seq1, seq2, match_score, mismatch_penalty, gap_penalty)

Матрица выравнивания 2 последовательностей:
[0, -2, -4, -6, -8, -10, -12]
[-2, 1, -1, -3, -5, -7, -9]
[-4, -1, 0, -2, -4, -6, -8]
[-6, -3, -2, 1, -1, -3, -5]
[-8, -5, -4, -1, 0, 0, -2]
[-10, -7, -6, -3, 0, -1, 1]
[-12, -9, -8, -5, -2, -1, 0]

Итоговое выравнивание - 2 последовательности с гэпами:
CGTCTT
CATTCT

Скор выравнивания 2 последовательностей: 0


('CGTCTT', 'CATTCT', 0)

In [4]:
# Пример использования 3
seq1 = "ATGTCAC"
seq2 = "ATCTCC"
match_score = 1
mismatch_penalty = -2
gap_penalty = -2

needleman_wunsch(seq1, seq2, match_score, mismatch_penalty, gap_penalty)

Матрица выравнивания 2 последовательностей:
[0, -2, -4, -6, -8, -10, -12]
[-2, 1, -1, -3, -5, -7, -9]
[-4, -1, 2, 0, -2, -4, -6]
[-6, -3, 0, 0, -2, -4, -6]
[-8, -5, -2, -2, 1, -1, -3]
[-10, -7, -4, -1, -1, 2, 0]
[-12, -9, -6, -3, -3, 0, 0]
[-14, -11, -8, -5, -5, -2, 1]

Итоговое выравнивание - 2 последовательности с гэпами:
ATGTCAC
ATCTC-C

Скор выравнивания 2 последовательностей: 1


('ATGTCAC', 'ATCTC-C', 1)

#### Как работает код:

```python
def needleman_wunsch(seq1, seq2, match_score, mismatch_penalty, gap_penalty):
    """
    seq1, seq2 - выравниваемые последовательности нуклеотидов
    match_score - балл за совпадение
    mismatch_penalty - штраф за несовпадение
    gap_penalty - штраф за вставку или удаление
    """
```
- **Определение функции**: Функция принимает две последовательности (`seq1` и `seq2`), балл за совпадение (`match_score`), штраф за несовпадение (`mismatch_penalty`) и штраф за вставку или удаление (`gap_penalty`).

```python
    n = len(seq1)
    m = len(seq2)
```
- **Инициализация длин последовательностей**: `n` и `m` хранят длины `seq1` и `seq2` соответственно.

```python
    score_matrix = [[0 for j in range(m + 1)] for i in range(n + 1)]
```
- **Создание матрицы оценок**: Матрица размером `(n+1) x (m+1)` инициализируется нулями. Эта матрица будет хранить максимальные баллы выравнивания для каждой позиции.

```python
    traceback_matrix = [['' for j in range(m + 1)] for i in range(n + 1)]
```
- **Создание матрицы трассировки**: Матрица того же размера, что и `score_matrix`, будет хранить указатели на предыдущие клетки, что поможет восстановить оптимальное выравнивание.

```python
    for i in range(1, n + 1):
        score_matrix[i][0] = gap_penalty * i
        traceback_matrix[i][0] = '↑'
    for j in range(1, m + 1):
        score_matrix[0][j] = gap_penalty * j
        traceback_matrix[0][j] = '←'
```
- **Инициализация первой строки и первого столбца**: Заполняем первую строку и первый столбец матрицы штрафами за пробелы. Указатели в матрице трассировки указывают на движение вверх (`↑`) и влево (`←`).

```python
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if seq1[i - 1] == seq2[j - 1]:
                diag = score_matrix[i - 1][j - 1] + match_score
            else:
                diag = score_matrix[i - 1][j - 1] + mismatch_penalty
            up = score_matrix[i - 1][j] + gap_penalty
            left = score_matrix[i][j - 1] + gap_penalty
```
- **Заполнение матрицы оценок**: Для каждой ячейки `[i][j]` вычисляем баллы для диагонального (`diag`), вертикального (`up`) и горизонтального (`left`) движений. Если символы в последовательностях совпадают, добавляем `match_score`, иначе `mismatch_penalty`.

```python
            max_score = max(diag, up, left)
            score_matrix[i][j] = max_score
```
- **Выбор максимального балла**: Из трех возможных движений выбираем максимальный балл и записываем его в `score_matrix`.

```python
            if max_score == diag:
                traceback_matrix[i][j] = '↖'
            elif max_score == up:
                traceback_matrix[i][j] = '↑'
            else:
                traceback_matrix[i][j] = '←'
```
- **Заполнение матрицы трассировки**: В зависимости от того, откуда пришел максимальный балл, записываем соответствующий указатель в `traceback_matrix`.

```python
    align1 = ''
    align2 = ''
    i = n
    j = m
```
- **Инициализация переменных для выравнивания**: Переменные `align1` и `align2` будут хранить выровненные последовательности. Начинаем трассировку с нижнего правого угла матрицы.

```python
    while i > 0 or j > 0:
        if traceback_matrix[i][j] == '↖':
            align1 = seq1[i - 1] + align1
            align2 = seq2[j - 1] + align2
            i -= 1
            j -= 1
        elif traceback_matrix[i][j] == '↑':
            align1 = seq1[i - 1] + align1
            align2 = '-' + align2
            i -= 1
        else:
            align1 = '-' + align1
            align2 = seq2[j - 1] + align2
            j -= 1
```
- **Трассировка для получения оптимального выравнивания**: Восстанавливаем путь, по которому мы пришли к клетке `[i][j]`. В зависимости от указателя в `traceback_matrix`, определяем движение и добавляем соответствующие символы или пробелы в `align1` и `align2`.

```python
    print("Матрица выравнивания 2 последовательностей:")
    for row in score_matrix:
        print(row)
```
- **Вывод матрицы оценок**: Печатаем матрицу выравнивания для наглядности.

```python
    print("\nИтоговое выравнивание - 2 последовательности с гэпами:")
    print(align1)
    print(align2)
```
- **Вывод итогового выравнивания**: Печатаем выровненные последовательности с пробелами.

```python
    print("\nСкор выравнивания 2 последовательностей:", score_matrix[n][m])
```
- **Вывод скор выравнивания**: Печатаем максимальный балл выравнивания, который находится в нижнем правом углу матрицы.

```python
    return align1, align2, score_matrix[n][m]
```
- **Возврат результатов**: Функция возвращает выровненные последовательности и максимальный балл выравнивания.


In [5]:
# Первый пример
s1 = 'GACGAAG'
s2 = 'ACCAAG'
match = 1
mismatch = -1
indel = -1
alignment = needleman_wunsch(s1, s2, match, mismatch, indel)
print('1) Выравнивание последовательностей:')
print(alignment[0])
print(alignment[1])
print('Общий счет:', alignment[2])
print()

# Второй пример
s1 = 'CGTCTT'
s2 = 'CATTCT'
match = 1
mismatch = -1
indel = -2
alignment = needleman_wunsch(s1, s2, match, mismatch, indel)
print('2) Выравнивание последовательностей:')
print(alignment[0])
print(alignment[1])
print('Общий счет:', alignment[2])
print()

# Третий пример
s1 = 'ATGTCAC'
s2 = 'ATCTCC'
match = 1
mismatch = -2
indel = -2
alignment = needleman_wunsch(s1, s2, match, mismatch, indel)
print('3) Выравнивание последовательностей:')
print(alignment[0])
print(alignment[1])
print('Общий счет:', alignment[2])

Матрица выравнивания 2 последовательностей:
[0, -1, -2, -3, -4, -5, -6]
[-1, -1, -2, -3, -4, -5, -4]
[-2, 0, -1, -2, -2, -3, -4]
[-3, -1, 1, 0, -1, -2, -3]
[-4, -2, 0, 0, -1, -2, -1]
[-5, -3, -1, -1, 1, 0, -1]
[-6, -4, -2, -2, 0, 2, 1]
[-7, -5, -3, -3, -1, 1, 3]

Итоговое выравнивание - 2 последовательности с гэпами:
GACGAAG
-ACCAAG

Скор выравнивания 2 последовательностей: 3
1) Выравнивание последовательностей:
GACGAAG
-ACCAAG
Общий счет: 3

Матрица выравнивания 2 последовательностей:
[0, -2, -4, -6, -8, -10, -12]
[-2, 1, -1, -3, -5, -7, -9]
[-4, -1, 0, -2, -4, -6, -8]
[-6, -3, -2, 1, -1, -3, -5]
[-8, -5, -4, -1, 0, 0, -2]
[-10, -7, -6, -3, 0, -1, 1]
[-12, -9, -8, -5, -2, -1, 0]

Итоговое выравнивание - 2 последовательности с гэпами:
CGTCTT
CATTCT

Скор выравнивания 2 последовательностей: 0
2) Выравнивание последовательностей:
CGTCTT
CATTCT
Общий счет: 0

Матрица выравнивания 2 последовательностей:
[0, -2, -4, -6, -8, -10, -12]
[-2, 1, -1, -3, -5, -7, -9]
[-4, -1, 2, 0, -2, -4, -6]
[-